In [1]:
# ═══════════════════════════════════════════════════════════
# NOTEBOOK 04 — AUTOMATED HYPERPARAMETER OPTIMIZATION
# Marathi Mitra — Optimized for Colab Pro A100
#
# Goal: Improve unseen word score beyond v2's 78.8%
# v2 baseline: seen=100%, unseen=78.8%, overall=89.4%
#
# Requirements: Colab Pro A100 (40GB VRAM)
# Expected time: ~40-60 minutes
# ═══════════════════════════════════════════════════════════

import torch

print("🔍 Marathi Mitra — Automated HPO")
print("=" * 60)
print(f"GPU:  {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print()

vram = torch.cuda.get_device_properties(0).total_memory / 1e9
IS_A100 = vram >= 35

if IS_A100:
    print("✅ A100 detected — full search enabled")
    print("   → 20 trials")
    print("   → r up to 64")
    print("   → 8-bit quantization option")
    print("   → batch size 4")
else:
    print("⚠️  T4 detected — reduced search")
    print("   → 5 trials max")
    print("   → r up to 32")
    print("   → 4-bit only")
    print("   → batch size 2")

print()
print("v2 baseline to beat:")
print("   Seen:   100.0%")
print("   Unseen:  78.8%  ← Optuna targets this")
print("   Overall: 89.4%")

🔍 Marathi Mitra — Automated HPO
GPU:  NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB

✅ A100 detected — full search enabled
   → 20 trials
   → r up to 64
   → 8-bit quantization option
   → batch size 4

v2 baseline to beat:
   Seen:   100.0%
   Unseen:  78.8%  ← Optuna targets this
   Overall: 89.4%


In [3]:
!pip install -q "numpy>=2.0"
!pip install -q transformers==4.44.0
!pip install -q datasets==2.20.0
!pip install -q peft==0.12.0
!pip install -q trl==0.9.6
!pip install -q bitsandbytes==0.45.3
!pip install -q accelerate==0.33.0
!pip install -q huggingface_hub
!pip install -q optuna

print("✅ All libraries installed!")
print("⚠️  Runtime → Restart Session → skip this cell → run Cell 3")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 158.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 39.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.5.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 27.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━

In [1]:
import os
import re
import json
import torch
import optuna
import urllib.request
from google.colab import userdata
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

# ── Credentials ───────────────────────────────────────────
HF_TOKEN     = userdata.get("HF_TOKEN")
HF_USERNAME  = "ninadp"
OPTUNA_REPO  = f"{HF_USERNAME}/marathi-mitra-phi3-v3"

login(token=HF_TOKEN)
print(f"✅ Logged in as: {HF_USERNAME}")
print(f"✅ Best model will save to: {OPTUNA_REPO}")

# ── Dataset ───────────────────────────────────────────────
GITHUB_USERNAME = "ninadparab"
url = (
    f"https://raw.githubusercontent.com/{GITHUB_USERNAME}"
    f"/marathi-mitra/main/data/vocabulary_dataset.json"
)

with urllib.request.urlopen(url) as response:
    raw_data = json.loads(response.read().decode("utf-8"))

hf_dataset = Dataset.from_list(raw_data)
print(f"✅ Dataset: {len(hf_dataset)} examples")

# ── Tokenizer (load once) ─────────────────────────────────
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
tokenizer  = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True
)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"✅ Tokenizer loaded")

✅ Logged in as: ninadp
✅ Best model will save to: ninadp/marathi-mitra-phi3-v3
✅ Dataset: 250 examples


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

✅ Tokenizer loaded


In [2]:
# ── Hardware limits ───────────────────────────────────────
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
IS_A100 = VRAM_GB >= 35

if IS_A100:
    N_TRIALS   = 20
    MAX_R      = 64
    MAX_EPOCHS = 40
    BATCH_SIZE = [2, 4]
    QUANT_OPT  = ["4bit"]
else:
    N_TRIALS   = 5
    MAX_R      = 32
    MAX_EPOCHS = 30
    BATCH_SIZE = [2]
    QUANT_OPT  = ["4bit"]

print(f"Hardware:  {'A100 ✅' if IS_A100 else 'T4 ⚠️'}")
print(f"Trials:    {N_TRIALS}")
print(f"Max r:     {MAX_R}")
print(f"Max epochs:{MAX_EPOCHS}")

# ── Unseen words — what Optuna optimizes for ──────────────
UNSEEN_WORDS = {
    "apple":  {"marathi": "सफरचंद", "pronunciation": "Sa-far-chand"},
    "star":   {"marathi": "तारा",    "pronunciation": "Taa-raa"},
    "tiger":  {"marathi": "वाघ",     "pronunciation": "Vaagh"},
    "ocean":  {"marathi": "समुद्र",  "pronunciation": "Sa-mu-dra"},
    "dance":  {"marathi": "नृत्य",   "pronunciation": "Nru-tya"},
}

# ── Seen words — also tracked but not optimized ───────────
SEEN_WORDS = {
    "butterfly": {"marathi": "फुलपाखरू", "pronunciation": "Phul-pakh-roo"},
    "mother":    {"marathi": "आई",        "pronunciation": "Aa-ee"},
    "rain":      {"marathi": "पाऊस",      "pronunciation": "Paa-oos"},
    "elephant":  {"marathi": "हत्ती",     "pronunciation": "Hat-tee"},
    "school":    {"marathi": "शाळा",      "pronunciation": "Shaa-la"},
}

# ── Store trial results ───────────────────────────────────
TRIAL_RESULTS = []

print(f"\nOptimizing for unseen words: {list(UNSEEN_WORDS.keys())}")
print(f"v2 baseline to beat: 78.8% unseen")

Hardware:  A100 ✅
Trials:    20
Max r:     64
Max epochs:40

Optimizing for unseen words: ['apple', 'star', 'tiger', 'ocean', 'dance']
v2 baseline to beat: 78.8% unseen


In [3]:
# ── Load base model ONCE at startup ──────────────────────
print("Loading base model once for all trials...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
BASE_MODEL_WEIGHTS = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
print("✅ Base model loaded once — reused across all trials")


# ── Generate lesson ───────────────────────────────────────
def generate(word, model, tokenizer):
    prompt = f"""### Instruction:
You are Marathi Mitra, a friendly Marathi teacher for kids. \
When given an English word, teach it in Marathi with the word \
in Devanagari script, pronunciation, a simple story sentence, \
and a fun fact. Always be encouraging and kid-friendly.

### Input:
Teach me the Marathi word for: {word}

### Response:
"""
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text.split("### Response:")[-1].strip()


# ── Extract fields ────────────────────────────────────────
def extract_fields(output):
    fields = {}
    marathi_match = re.search(r"is \*\*([^*]+)\*\*", output)
    fields["marathi_word"] = (
        marathi_match.group(1).strip() if marathi_match else ""
    )
    pronun_match = re.search(r"How to say it:\*?\*?\s*(.+)", output)
    fields["pronunciation"] = (
        pronun_match.group(1).strip() if pronun_match else ""
    )
    return fields


# ── Score one word ────────────────────────────────────────
def score_word(output, expected_marathi, expected_pronun):
    fields = extract_fields(output)

    presence = {
        "Has Devanagari":    bool(re.search(r"[\u0900-\u097F]", output)),
        "Has pronunciation": "How to say" in output,
        "Has sentence":      "Example sentence" in output,
        "Has fun fact":      "Fun Fact" in output,
        "Has emojis":        "🌟" in output and "📢" in output,
    }
    presence_score = sum(presence.values()) / len(presence) * 100

    exact = {}
    exact["Marathi correct"] = (
        expected_marathi in fields["marathi_word"] or
        expected_marathi in output
    )
    m = fields["pronunciation"].lower().replace(" ", "").replace("-", "")
    g = expected_pronun.lower().replace(" ", "").replace("-", "")
    exact["Pronunciation correct"] = m == g
    exact_score = sum(exact.values()) / len(exact) * 100

    return presence_score * 0.40 + exact_score * 0.60


# ── Evaluate on word set ──────────────────────────────────
def evaluate_words(model, tokenizer, word_dict):
    total = 0
    for word, golden in word_dict.items():
        try:
            output = generate(word, model, tokenizer)
            score  = score_word(
                output,
                golden["marathi"],
                golden["pronunciation"],
            )
            total += score
        except Exception as e:
            print(f"  Error on {word}: {e}")
    return total / len(word_dict)


print("✅ Helper functions ready")

✅ Helper functions ready


In [4]:
def objective(trial):
    """
    Optuna calls this once per trial.
    Optimizes for UNSEEN word score to improve generalisation.
    Warm started with v2 best config (r=32, lr=2e-4, epochs=25).
    """

    # ── Suggest hyperparameters ───────────────────────────
    # Search around v2's best config
    learning_rate = trial.suggest_float(
        "learning_rate",
        5e-5, 3e-4,       # narrowed around 2e-4 that worked
        log=True,
    )

    epochs = trial.suggest_int(
        "epochs",
        20, MAX_EPOCHS,   # around 25 that worked
    )

    r = trial.suggest_categorical(
        "r",
        [16, 32, 64] if IS_A100 else [16, 32],
    )

    lora_alpha = r * 2

    quantization = trial.suggest_categorical(
        "quantization", QUANT_OPT,
    )

    batch_size = trial.suggest_categorical(
        "batch_size", BATCH_SIZE,
    )

    print(f"\n{'─'*60}")
    print(f"Trial {trial.number + 1}/{N_TRIALS}")
    print(f"  lr={learning_rate:.2e}  epochs={epochs}")
    print(f"  r={r}  alpha={lora_alpha}")
    print(f"  quantization={quantization}  batch={batch_size}")
    print(f"{'─'*60}")

    # ── Train ─────────────────────────────────────────────
    # ── Apply fresh LoRA on existing base model ───────────
    # No need to reload Phi-3 every trial!
    from peft import get_peft_model, prepare_model_for_kbit_training
    import copy

    # Deep copy base model for this trial
    model = copy.deepcopy(BASE_MODEL_WEIGHTS)
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    # Train/eval split
    split          = hf_dataset.train_test_split(test_size=0.2, seed=42)
    train_dataset  = split["train"]
    eval_dataset   = split["test"]

    training_args = SFTConfig(
        output_dir=f"./optuna_trial_{trial.number}",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4 if batch_size == 2 else 2,
        learning_rate=learning_rate,
        fp16=True,
        logging_steps=10,
        max_seq_length=512,
        dataset_text_field="text",
        warmup_ratio=0.1,
        save_strategy="no",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
    )

    trainer.train()

    step_losses = [
        log["loss"] for log in trainer.state.log_history
        if "loss" in log and "eval_loss" not in log
    ]
    final_loss = step_losses[-1] if step_losses else 99
    print(f"  Final train loss: {final_loss:.4f}")

    # ── Evaluate on UNSEEN words ──────────────────────────
    unseen_score = evaluate_words(model, tokenizer, UNSEEN_WORDS)
    seen_score   = evaluate_words(model, tokenizer, SEEN_WORDS)
    print(f"  Seen score:   {seen_score:.1f}%")
    print(f"  Unseen score: {unseen_score:.1f}%  ← Optuna optimizes this")

    # ── Store result ──────────────────────────────────────
    TRIAL_RESULTS.append({
        "trial":         trial.number + 1,
        "learning_rate": learning_rate,
        "epochs":        epochs,
        "r":             r,
        "lora_alpha":    lora_alpha,
        "quantization":  quantization,
        "batch_size":    batch_size,
        "final_loss":    round(final_loss, 4),
        "seen_score":    round(seen_score, 1),
        "unseen_score":  round(unseen_score, 1),
        "overall":       round((seen_score + unseen_score) / 2, 1),
        "model":         model,
    })

    # Free GPU memory
    del trainer
    torch.cuda.empty_cache()

    return unseen_score


print("✅ Objective function defined")
print(f"   Optimizing for: unseen word score")
print(f"   Baseline to beat: 78.8%")

✅ Objective function defined
   Optimizing for: unseen word score
   Baseline to beat: 78.8%


In [ ]:
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Create study ──────────────────────────────────────────
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="marathi-mitra-v3-hpo",
)

# ── Warm start with manual experiment results ─────────────
# Optuna learns from these before starting new trials
print("Warm starting with known results...")

manual_results = [
    # exp1 — epochs=5 is outside range, clip to 20
    ({"learning_rate": 2e-4, "epochs": 20,  "r": 16,
      "quantization": "4bit", "batch_size": 2}, 25.6),
    # exp2
    ({"learning_rate": 2e-4, "epochs": 25, "r": 16,
      "quantization": "4bit", "batch_size": 2}, 25.6),
    # exp3
    ({"learning_rate": 1e-4, "epochs": 25, "r": 16,
      "quantization": "4bit", "batch_size": 2}, 25.6),
    # v2 best ← most important
    ({"learning_rate": 2e-4, "epochs": 25, "r": 32,
      "quantization": "4bit", "batch_size": 2}, 78.8),
]

for params, score in manual_results:
    trial = optuna.trial.create_trial(
        params=params,
        distributions={
            "learning_rate": optuna.distributions.FloatDistribution(
                5e-5, 3e-4, log=True
            ),
            "epochs": optuna.distributions.IntDistribution(20, MAX_EPOCHS),
            "r": optuna.distributions.CategoricalDistribution(
                [16, 32, 64] if IS_A100 else [16, 32]
            ),
            "quantization": optuna.distributions.CategoricalDistribution(
                ["4bit"]
            ),
            "batch_size": optuna.distributions.CategoricalDistribution(
                BATCH_SIZE
            ),
        },
        value=score,
    )
    study.add_trial(trial)

print(f"✅ Warm started with {len(manual_results)} known results")
print(f"   Optuna knows r=32, lr=2e-4 → 78.8% unseen")
print(f"   Will search around and beyond this")
print(f"\n🔍 Starting {N_TRIALS} trials...")
print(f"   Estimated time: ~{N_TRIALS * 2}-{N_TRIALS * 3} mins on A100")
print(f"{'═'*60}")

# ── Run study ─────────────────────────────────────────────
study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=False,
    gc_after_trial=True,
)

print(f"\n{'═'*60}")
print(f"✅ Optuna search complete!")
print(f"   Best unseen score: {study.best_value:.1f}%")
print(f"   Best params: {study.best_params}")

Warm starting with known results...
✅ Warm started with 4 known results
   Optuna knows r=32, lr=2e-4 → 78.8% unseen
   Will search around and beyond this

🔍 Starting 20 trials...
   Estimated time: ~40-60 mins on A100
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
Trial 5/20
  lr=9.78e-05  epochs=39
  r=16  alpha=32
  quantization=4bit  batch=2
────────────────────────────────────────────────────────────


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py: 0.00B [00:00, ?B/s]

...

  Final train loss: 0.0882
  Seen score:   32.0%
  Unseen score: 40.8%  ← Optuna optimizes this

────────────────────────────────────────────────────────────
Trial 10/20
  lr=2.13e-04  epochs=26
  r=32  alpha=64
  quantization=4bit  batch=4
────────────────────────────────────────────────────────────


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Step,Training Loss
10,1.760400
20,1.621000
30,1.212700
40,0.616500
50,0.481100
60,0.435500
70,0.407300
80,0.385300
90,0.363900
100,0.348500


In [ ]:
# ── Sort by unseen score ──────────────────────────────────
sorted_results = sorted(
    TRIAL_RESULTS,
    key=lambda x: x["unseen_score"],
    reverse=True,
)

print(f"\n{'═'*80}")
print("OPTUNA TRIAL RESULTS")
print(f"{'═'*80}")
print(
    f"{'Trial':<7} {'LR':<10} {'Ep':<5} {'r':<5} "
    f"{'Quant':<7} {'BS':<4} {'Loss':<8} {'Seen':>7} {'Unseen':>8} {'Overall':>8}"
)
print("─" * 80)

for result in sorted_results:
    print(
        f"{result['trial']:<7} "
        f"{result['learning_rate']:<10.2e} "
        f"{result['epochs']:<5} "
        f"{result['r']:<5} "
        f"{result['quantization']:<7} "
        f"{result['batch_size']:<4} "
        f"{result['final_loss']:<8} "
        f"{result['seen_score']:>6.1f}% "
        f"{result['unseen_score']:>7.1f}% "
        f"{result['overall']:>7.1f}%"
    )

print(f"{'═'*80}")
print(f"\n🏆 Best trial: {study.best_trial.number + 1}")
print(f"   Unseen score: {study.best_value:.1f}%")
print(f"   Params: {study.best_params}")

In [ ]:
print(f"\n{'═'*60}")
print("OPTUNA vs PREVIOUS RESULTS")
print(f"{'═'*60}")

print(f"\n{'Version':<20} {'Seen':>8} {'Unseen':>8} {'Overall':>8}")
print("─" * 50)
print(f"{'Base Phi-3':<20} {'11.2%':>8} {'8.0%':>8} {'9.6%':>8}")
print(f"{'v1 (30 examples)':<20} {'36.4%':>8} {'25.6%':>8} {'31.0%':>8}")
print(f"{'v2 (250 examples)':<20} {'100.0%':>8} {'78.8%':>8} {'89.4%':>8}")

# Best Optuna result
best_result = sorted_results[0]
print(
    f"{'v3 (Optuna)':<20} "
    f"{best_result['seen_score']:>7.1f}% "
    f"{best_result['unseen_score']:>7.1f}% "
    f"{best_result['overall']:>7.1f}%"
)

print(f"\n📈 Improvement over v2:")
unseen_improvement = best_result["unseen_score"] - 78.8
overall_improvement = best_result["overall"] - 89.4
print(f"   Unseen: {unseen_improvement:+.1f}%")
print(f"   Overall: {overall_improvement:+.1f}%")

if unseen_improvement > 5:
    print("\n✅ Optuna found significantly better config!")
elif unseen_improvement > 0:
    print("\n✅ Optuna improved generalisation")
else:
    print("\n⚠️  v2 config was already near optimal")
    print("   More diverse training data would help more than HPO")

In [ ]:
import os

# ── Find best model ───────────────────────────────────────
best_trial_num  = study.best_trial.number
best_trial_data = next(
    r for r in TRIAL_RESULTS
    if r["trial"] == best_trial_num + 1
)
best_model = best_trial_data["model"]

print(f"Saving best model (Trial {best_trial_num + 1})...")
print(f"  Unseen score: {best_trial_data['unseen_score']:.1f}%")
print(f"  Overall:      {best_trial_data['overall']:.1f}%")
print(f"  Repo:         {OPTUNA_REPO}")

# ── Save locally first ────────────────────────────────────
local_path = "./results/optuna_best"
os.makedirs(local_path, exist_ok=True)

best_model.save_pretrained(local_path)
tokenizer.save_pretrained(local_path)

files = os.listdir(local_path)
print(f"\nFiles saved: {files}")
assert "adapter_config.json" in files, "❌ Missing adapter_config.json!"
print("✅ adapter_config.json present")

# ── Push to HF Hub ────────────────────────────────────────
print(f"\nPushing to {OPTUNA_REPO}...")
best_model.push_to_hub(
    OPTUNA_REPO,
    token=HF_TOKEN,
    commit_message=(
        f"v3 Optuna best: unseen={best_trial_data['unseen_score']:.1f}% "
        f"overall={best_trial_data['overall']:.1f}%"
    ),
)
tokenizer.push_to_hub(OPTUNA_REPO, token=HF_TOKEN)

print(f"\n✅ v3 model saved!")
print(f"✅ https://huggingface.co/{OPTUNA_REPO}")

In [ ]:
best_params = study.best_params

config_content = f"""# ═══════════════════════════════════════════════════════════
# config.yaml — Marathi Mitra
# Best hyperparameters found via Optuna (v3)
# Unseen score: {study.best_value:.1f}%
# ═══════════════════════════════════════════════════════════

model:
  name: "microsoft/Phi-3-mini-4k-instruct"
  max_seq_length: 512

lora:
  r: {best_params['r']}
  alpha: {best_params['r'] * 2}
  dropout: 0.05
  target_modules:
    - q_proj
    - k_proj
    - v_proj
    - o_proj

training:
  learning_rate: {best_params['learning_rate']}
  epochs: {best_params['epochs']}
  batch_size: {best_params['batch_size']}
  gradient_accumulation_steps: 4
  warmup_ratio: 0.1
  fp16: true
  logging_steps: 10
  output_dir: "./results/best_model"
  quantization: "{best_params['quantization']}"
  train_test_split: 0.2

data:
  vocabulary_file: "./data/vocabulary.json"
  dataset_file:    "./data/vocabulary_dataset.json"
  text_field:      "text"

huggingface:
  model_repo: "{OPTUNA_REPO}"

optuna:
  n_trials: {N_TRIALS}
  hardware: "{'A100' if IS_A100 else 'T4'}"
  optimized_for: "unseen_word_score"
  baseline_v2_unseen: 78.8
  best_unseen: {study.best_value:.1f}
  best_trial: {study.best_trial.number + 1}
"""

with open("config.yaml", "w") as f:
    f.write(config_content)

print("✅ config.yaml updated!")
print()
print(config_content)

In [ ]:
print(f"{'═'*60}")
print("NOTEBOOK 04 — COMPLETE SUMMARY")
print(f"{'═'*60}")

best = sorted_results[0]

print(f"""
Hardware:        {'A100 (40GB)' if IS_A100 else 'T4 (15GB)'}
Trials run:      {N_TRIALS}
Warm started:    4 manual experiments
Optimized for:   unseen word generalisation

Best config found:
  learning_rate: {study.best_params['learning_rate']:.2e}
  epochs:        {study.best_params['epochs']}
  r:             {study.best_params['r']}
  lora_alpha:    {study.best_params['r'] * 2}
  quantization:  {study.best_params['quantization']}
  batch_size:    {study.best_params['batch_size']}

Results:
  Seen score:    {best['seen_score']:.1f}%
  Unseen score:  {best['unseen_score']:.1f}%   (v2 was 78.8%)
  Overall:       {best['overall']:.1f}%         (v2 was 89.4%)

Model saved:     huggingface.co/{OPTUNA_REPO}
Config updated:  config.yaml

Next steps:
→ Update app.py MODEL_REPO to {OPTUNA_REPO}
→ Push to HF Spaces → redeploy
→ Update README with v3 results
→ Ctrl+S → Save notebook to GitHub
""")